# Comparativo de Modelos — TriageAI

Este notebook lê todos os runs do experimento `triageai-baseline-oficial` no MLflow, plota um comparativo de métricas e justifica a escolha do modelo **campeão**.

**Pré-requisito:** rodar primeiro `treinamento_baseline.ipynb` e `treinamento_xgboost.ipynb` para ter ao menos 2 runs registrados.

In [1]:
import os
import mlflow
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # backend sem GUI para ambiente Docker
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Conexão com MLflow
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000")
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow conectado em: {tracking_uri}")

MLflow conectado em: http://mlflow:5000


In [2]:
# Carrega todos os runs do experimento
EXPERIMENT_NAME = "triageai-baseline-oficial"

df_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.f1_score DESC"],
)

print(f"Runs encontrados: {len(df_runs)}")

if df_runs.empty:
    raise RuntimeError(
        "Nenhum run encontrado. Execute treinamento_baseline.ipynb e "
        "treinamento_xgboost.ipynb antes de rodar este notebook."
    )

# Seleciona colunas relevantes
cols = [
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.f1_score",
    "metrics.precision_weighted",
    "metrics.recall_weighted",
    "params.model_type",
    "params.vectorizer",
    "status",
]
cols_existentes = [c for c in cols if c in df_runs.columns]
df_view = df_runs[cols_existentes].copy()
df_view = df_view[df_view["status"] == "FINISHED"].reset_index(drop=True)
df_view.rename(columns={"tags.mlflow.runName": "run_name"}, inplace=True)

print("\nRuns concluídos:")
display(df_view)

Runs encontrados: 5

Runs concluídos:


,run_name,metrics.accuracy,metrics.f1_score,metrics.precision_weighted,metrics.recall_weighted,params.model_type,params.vectorizer,status
0,xgboost_dataset_gold,0.874736,0.875176,0.876698,0.874736,XGBoost,TF-IDF,FINISHED


In [3]:
# Gráfico de barras — Acurácia e F1-Score por modelo
metricas = ["metrics.accuracy", "metrics.f1_score"]
metricas_existentes = [m for m in metricas if m in df_view.columns]

if not metricas_existentes:
    print("Nenhuma métrica de accuracy/f1_score encontrada nos runs.")
else:
    labels = df_view["run_name"].tolist()
    x = range(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 2.5), 5))

    bars1, bars2 = [], []
    if "metrics.accuracy" in df_view.columns:
        bars1 = ax.bar(
            [xi - width / 2 for xi in x],
            df_view["metrics.accuracy"].fillna(0),
            width,
            label="Acurácia",
            color="#2196F3",
            alpha=0.85,
        )
    if "metrics.f1_score" in df_view.columns:
        bars2 = ax.bar(
            [xi + width / 2 for xi in x],
            df_view["metrics.f1_score"].fillna(0),
            width,
            label="F1-Score (weighted)",
            color="#00843D",
            alpha=0.85,
        )

    ax.set_xlabel("Modelo / Run")
    ax.set_ylabel("Valor")
    ax.set_title(f"Comparativo de Modelos — Experimento: {EXPERIMENT_NAME}")
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    for bar in (list(bars1) + list(bars2)):
        h = bar.get_height()
        ax.annotate(
            f"{h:.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            fontsize=9,
        )

    plt.tight_layout()
    caminho_grafico = "/tmp/comparativo_modelos.png"
    plt.savefig(caminho_grafico, dpi=120)
    plt.show()
    print(f"Gráfico salvo em: {caminho_grafico}")

Gráfico salvo em: /tmp/comparativo_modelos.png


In [4]:
# Identifica o modelo campeão (maior F1-Score weighted)
if "metrics.f1_score" in df_view.columns:
    idx_campeao = df_view["metrics.f1_score"].idxmax()
    campeao = df_view.loc[idx_campeao]

    print("\n" + "="*60)
    print("MODELO CAMPEÃO")
    print("="*60)
    print(f"  Run:        {campeao.get('run_name', 'N/A')}")
    print(f"  Modelo:     {campeao.get('params.model_type', 'N/A')}")
    print(f"  Acurácia:   {campeao.get('metrics.accuracy', 'N/A'):.4f}")
    print(f"  F1-Score:   {campeao.get('metrics.f1_score', 'N/A'):.4f}")
    print()
    print("Justificativa de escolha:")
    print("  O critério de seleção é o F1-Score weighted, que pondera a")
    print("  performance por classe levando em conta o desbalanceamento")
    print("  natural do dataset (algumas doenças têm muito mais amostras).")
    print("  Acurácia isolada é enganosa em datasets desbalanceados.")
else:
    print("Métrica f1_score não encontrada — verifique os runs registrados.")


MODELO CAMPEÃO
  Run:        xgboost_dataset_gold
  Modelo:     XGBoost
  Acurácia:   0.8747
  F1-Score:   0.8752

Justificativa de escolha:
  O critério de seleção é o F1-Score weighted, que pondera a
  performance por classe levando em conta o desbalanceamento
  natural do dataset (algumas doenças têm muito mais amostras).
  Acurácia isolada é enganosa em datasets desbalanceados.


In [5]:
# Registra o gráfico e este notebook como artefatos MLflow
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="comparativo_modelos"):
    # Gráfico
    try:
        mlflow.log_artifact(caminho_grafico, artifact_path="comparativo")
        print(f"Gráfico registrado no MLflow.")
    except Exception as e:
        print(f"Aviso: não foi possível registrar gráfico — {e}")

    # Loga a tabela de comparação como CSV
    csv_path = "/tmp/comparativo_runs.csv"
    df_view.to_csv(csv_path, index=False)
    mlflow.log_artifact(csv_path, artifact_path="comparativo")

    # Loga o campeão como parâmetro
    if "metrics.f1_score" in df_view.columns:
        mlflow.log_param("campeao_run", campeao.get("run_name", "N/A"))
        mlflow.log_param("campeao_modelo", campeao.get("params.model_type", "N/A"))
        mlflow.log_metric("campeao_f1_score", campeao.get("metrics.f1_score", 0))

    print("\n✅ Comparativo registrado no MLflow com sucesso.")

Gráfico registrado no MLflow.

✅ Comparativo registrado no MLflow com sucesso.
